<a href="https://colab.research.google.com/github/Meralath/voice-monsterizer/blob/main/monsterize_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install librosa numpy soundfile

In [ ]:
import os

import librosa
import numpy as np
import soundfile as sf

INPUT_FILE = "vox.wav"
SAMPLE_RATE = 44100
PITCH_SEMITONES = -10  # negative = deeper
ROUGHNESS = 1.9  # 1.0 = no distortion, higher = more growl

In [ ]:
def pitch_down(y, sr, semitones):
    """shifts the pitch by the given semitones. negative = deeper"""
    y_shifted = librosa.effects.pitch_shift(y, sr=sr, n_steps=semitones)
    return y_shifted


def add_growl(y, roughness):
    """turns the signal up by roughness and clips it at -1/1 for the distortion"""
    y_monster = np.clip(y * roughness, -1.0, 1.0)  # anything past -1/1 gets cut off
    return y_monster


def monster_path(file_path):
    """makes the output file name, vox.wav -> vox_monster.wav"""
    base_path, _ = os.path.splitext(file_path)
    output_path = f"{base_path}_monster.wav"
    return output_path


def show_result(file_path, output_path, duration, semitones, roughness, sr):
    """prints what got processed and with which settings"""
    print(f"{file_path} -> {output_path}")
    print(f"{duration:.2f} s, {semitones:+g} semitones, roughness {roughness}, {sr} Hz")

In [ ]:
if not os.path.exists(INPUT_FILE):
    print(f"can't find {INPUT_FILE}")

else:
    y, sr = librosa.load(INPUT_FILE, sr=SAMPLE_RATE, mono=True)

    if len(y) == 0:
        print("the file is empty, no samples in it")

    else:
        duration = len(y) / sr

        y_shifted = pitch_down(y, sr, PITCH_SEMITONES)
        y_monster = add_growl(y_shifted, ROUGHNESS)

        output_path = monster_path(INPUT_FILE)
        sf.write(output_path, y_monster, sr)

        show_result(INPUT_FILE, output_path, duration, PITCH_SEMITONES, ROUGHNESS, sr)

some notes on the settings:

sample rate is 44100 by default, same as CD quality.

pitch semitones is -8 by default. negative makes it deeper, around -6 is a bit deep and -12 is really deep.

roughness is 1.5 by default, it's basically the distortion. 1 means no distortion and 2 or more gives a heavy growl.

if it sounds too robotic try -6 for the pitch. if there's not enough growl turn roughness up to 2 or more, and if it's too distorted bring it down to 1.2 or 1.3.